# DISTIL_STAR V2 - Single-model distillation from the full 34-model ensemble
Student = STAR architecture (Spectral Transformer with Anchored Residuals).
Teacher = ensemble median of the 34 supervised surrogates
(precision V2: 12 + CSON: 5 + CNP: 5 + STAR: 7 + DISTIL V1: 3 + SSST: 1 + NEPTUNE: 1),
precomputed offline by code/precompute_ensemble_targets_34.py.
Loss = alpha * MSE(pred, median) + (1-alpha) * kink_weighted_mse(pred, raw_data),
alpha = 0.70.
High-epoch schedule: 3000 epochs per seed (2x V1), 5 seeds, with a wall-clock
budget guard between seeds so completed seeds are always saved.

In [ ]:
EPOCHS = 3000
BATCH_SIZE = 16
LR = 2e-4
N_SEEDS = 5
SEED_BASE = 42
TIME_BUDGET_H = 7.5
CTX_NOISE = 0.02
ALPHA_DISTIL = 0.70
N_CHEB = 64
N_COS = 32
D_MODEL = 384
N_LAYERS = 8
N_HEADS = 12
N_E_FREQ = 32
N_K_FREQ = 16
LAM_SPEC = 5e-5
LAM_SMOOTH = 5e-4

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                      '--index-url', 'https://download.pytorch.org/whl/cu121',
                      'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
# V2: the training target is the 34-model median. It is shipped in the
# tcml-median34 dataset as ensemble_median_targets_34.npz and copied over
# the filename the stock trainer reads, so scripts/train_distil_pro.py
# needs no modification.
hits = list(INPUT.rglob('ensemble_median_targets_34.npz'))
assert hits, '34-model median targets not found in /kaggle/input'
median34 = hits[0]
median_dst = REPO_DIR / 'data' / 'processed' / 'ensemble_median_targets.npz'
median_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(median34, median_dst)
print('Setup ready, REPO_DIR =', REPO_DIR)
print('Teacher target = 34-model ensemble median:', median34)

In [ ]:
OUT_DIR = WORK / 'runs' / 'distil_v2'
OUT_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.time()
test_npz_ref = None
for i in range(N_SEEDS):
    elapsed_h = (time.time() - t0) / 3600.0
    if elapsed_h > TIME_BUDGET_H:
        print(f'== time budget reached ({elapsed_h:.2f} h), stopping at seed index {i} ==')
        break
    seed = SEED_BASE + i
    args = [sys.executable, 'scripts/train_distil_pro.py',
            '--epochs', str(EPOCHS),
            '--batch', str(BATCH_SIZE),
            '--lr', str(LR),
            '--n_seeds', '1',
            '--seed_base', str(seed),
            '--ctx_noise', str(CTX_NOISE),
            '--alpha_distil', str(ALPHA_DISTIL),
            '--n_cheb', str(N_CHEB),
            '--n_cos', str(N_COS),
            '--d_model', str(D_MODEL),
            '--n_layers', str(N_LAYERS),
            '--n_heads', str(N_HEADS),
            '--n_E_freq', str(N_E_FREQ),
            '--n_k_freq', str(N_K_FREQ),
            '--lam_spec', str(LAM_SPEC),
            '--lam_smooth', str(LAM_SMOOTH),
            '--out_root', str(OUT_DIR)]
    print('>>>', ' '.join(args))
    ts = time.time()
    rc = subprocess.call(args, cwd=str(REPO_DIR))
    print(f'<<< seed {seed} exit={rc} elapsed={(time.time()-ts)/60:.1f} min')
    assert rc == 0, f'training failed on seed {seed}'
    if i == 0:
        # Keep the seed-42 normalisation stats as the reference test_branches.npz
        # (same convention as V1, where the first seed wrote the file).
        test_npz_ref = (OUT_DIR / 'test_branches.npz').read_bytes()
if test_npz_ref is not None:
    (OUT_DIR / 'test_branches.npz').write_bytes(test_npz_ref)
print(f'Total elapsed: {(time.time()-t0)/3600.0:.2f} h')

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')